In [ ]:
import fastf1
import pandas as pd
import numpy as np
import time

print("\n✅ All libraries loaded successfully!")


✅ All libraries loaded successfully!


In [2]:
years = [2023, 2024, 2025]
all_quali = []
all_results = []

fastf1.Cache.enable_cache('../cache')

for y in years:
    for i in range(1, 25):
        try: 
            session = fastf1.get_session(y, i, 'Q')
            session.load(laps=True, telemetry=False, weather=False, messages=False)
            best_laps = session.laps[session.laps['IsPersonalBest'] == True].copy()
            best_laps = best_laps.groupby('Driver')['LapTime'].min().reset_index()
            best_laps.columns = ['Driver', 'BestQualiTime']
            best_laps['BestQualiTime'] = best_laps['BestQualiTime'].dt.total_seconds()
            best_laps['Year'] = y
            best_laps['Round'] = i
            driver_map = session.results[['Abbreviation', 'FullName']].set_index('Abbreviation')['FullName'].to_dict()
            best_laps['FullName'] = best_laps['Driver'].map(driver_map)
            best_laps = best_laps.drop(columns=['Driver'])
            all_quali.append(best_laps)
            time.sleep(1)  # Sleep to avoid hitting API rate limits
        except Exception as e:
                print(f"Skipping {y} Round {i}: {e}")
df_qual = pd.concat(all_quali, ignore_index=True)

for y in years:
        for i in range(1, 25):
            try:
                session = fastf1.get_session(y, i, 'R')
                session.load(laps=False, telemetry=False, weather=False, messages=False)
                result = session.results[['FullName', 'TeamName', 'GridPosition', 'Position']].copy()
                result['Round'] = i
                result['Year'] = y
                result['SessionType'] = 'R'
                result['Location'] = session.event['Location']
                all_results.append(result)
            except Exception as e:
                print(f"Skipping {y} Round {i}: {e}")
df_race = pd.concat(all_results, ignore_index=True)   

df = df_race.merge(df_qual, on=['FullName', 'Round', 'Year'], how='left')
df['Podium'] = (df['Position'] <= 3).astype(int)
print(df.shape)
print(df.isnull().sum())

core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '16', '55', '14', '63', '44', '18', '31', '27', '4', '77', '24', '22', '23', '2', '20', '81', '21', '10']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INF

Skipping 2023 Round 23: Invalid round: 23
Skipping 2023 Round 24: Invalid round: 24


core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '63', '55', '11', '14', '4', '81', '44', '27', '22', '18', '23', '3', '20', '77', '24', '2', '31', '10']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO

Skipping 2023 Round 23: Invalid round: 23
Skipping 2023 Round 24: Invalid round: 24


core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '16', '81', '14', '63', '38', '4', '44', '27', '23', '20', '31', '2', '22', '3', '77', '24', '18', '10']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 19 drivers: ['55', '16', '4', '81', '11', '18', '22', '14', '27', '20', '23', '3', '10', '77', '24', '31', '63', '44', '1']
core           INFO 	Loading data for Japanese Grand Prix - Race [v3.8.1]
req            INFO 	Using

(1398, 10)
FullName          0
TeamName          0
GridPosition      1
Position          1
Round             0
Year              0
SessionType       0
Location          0
BestQualiTime    10
Podium            0
dtype: int64


In [3]:
df['BestQualiTime'] = df.groupby(['Round', 'Year'])['BestQualiTime'].transform(
    lambda x: x.fillna(x.median())
)
df = df.dropna(subset=['Position', 'GridPosition'])
print(df.shape)  
print(df.isnull().sum())
print(df.shape)

(1397, 10)
FullName         0
TeamName         0
GridPosition     0
Position         0
Round            0
Year             0
SessionType      0
Location         0
BestQualiTime    0
Podium           0
dtype: int64
(1397, 10)


In [4]:
track_type = {
    # Street circuits — easy to confirm
    'Jeddah': 'street',
    'Baku': 'street',
    'Miami': 'street',
    'Monaco': 'street',
    'Marina Bay': 'street',
    'Las Vegas': 'street',
    'Melbourne': 'street',
    'Miami Gardens': 'street',
    'Madrid': 'street',

    # Permanent circuits
    'Sakhir': 'permanent',
    'Barcelona': 'permanent',
    'Montréal': 'permanent',
    'Spielberg': 'permanent',
    'Silverstone': 'permanent',
    'Budapest': 'permanent',
    'Spa-Francorchamps': 'permanent',
    'Zandvoort': 'permanent',
    'Monza': 'permanent',
    'Suzuka': 'permanent',
    'Lusail': 'permanent',
    'Austin': 'permanent',
    'Mexico City': 'permanent',
    'São Paulo': 'permanent',
    'Yas Island': 'permanent',
    'Shanghai': 'permanent',
    'Imola': 'permanent',
}

df['TrackType'] = df['Location'].map(track_type)
print(df['TrackType'].isnull().sum())  # should be 0
print(df['TrackType'].value_counts())

0
TrackType
permanent    979
street       418
Name: count, dtype: int64


C:\Users\Aakash\AppData\Local\Temp\ipykernel_1564\714422160.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['TrackType'] = df['Location'].map(track_type)


In [14]:
new_results = []
new_quali = []
year = 2026
rounds = [1, 2]

for i in rounds:
    try:
        # Race
        session = fastf1.get_session(year, i, 'R')
        session.load(laps=False, telemetry=False, weather=False, messages=False)
        result = session.results[['FullName', 'TeamName', 'GridPosition', 'Position']].copy()
        result['Round'] = i
        result['Year'] = year
        result['SessionType'] = 'R'
        result['Location'] = session.event['Location']
        new_results.append(result)
    except Exception as e:
        print(f"Skipping {year} R{i} Race: {e}")

    try:
        # Quali
        session = fastf1.get_session(year, i, 'Q')
        session.load(laps=True, telemetry=False, weather=False, messages=False)
        best_laps = session.laps[session.laps['IsPersonalBest'] == True].copy()
        best_laps = best_laps.groupby('Driver')['LapTime'].min().reset_index()
        best_laps.columns = ['Driver', 'BestQualiTime']
        best_laps['BestQualiTime'] = best_laps['BestQualiTime'].dt.total_seconds()
        best_laps['Round'] = i
        best_laps['Year'] = year
        driver_map = session.results[['Abbreviation', 'FullName']].set_index('Abbreviation')['FullName'].to_dict()
        best_laps['FullName'] = best_laps['Driver'].map(driver_map)
        best_laps = best_laps.drop(columns=['Driver'])
        new_quali.append(best_laps)
    except Exception as e:
        print(f"Skipping {year} R{i} Quali: {e}")

df_new_race = pd.concat(new_results, ignore_index=True)
df_new_qual = pd.concat(new_quali, ignore_index=True)

df_new = df_new_race.merge(df_new_qual, on=['FullName', 'Round', 'Year'], how='left')
df_new['Podium'] = (df_new['Position'] <= 3).astype(int)

# Add TrackType
df_new['TrackType'] = df_new['Location'].map(track_type)

# Append to existing df
df = pd.concat([df, df_new], ignore_index=True)
print(df.shape)

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]


req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 22 drivers: ['63', '12', '16', '44', '1', '3', '87', '41', '5', '10', '31', '23', '30', '43', '55', '11', '18', '14', '77', '6', '81', '27']
core           INFO 	Loading data for Australian Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	No lap data for driver 18
core        WARNING 	No lap data for driver 55
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 18)
core        WARNI

(1463, 11)


In [15]:
df['BestQualiTime'] = df.groupby(['Round', 'Year'])['BestQualiTime'].transform(
    lambda x: x.fillna(x.median())
)

In [16]:
df = df.sort_values(['Year', 'Round']).reset_index(drop=True)


In [17]:
df.to_csv('../data/f1_dataset_clean.csv', index=False)
print(df.shape)
print(df.dtypes)

(1463, 11)
FullName          object
TeamName          object
GridPosition     float64
Position         float64
Round              int64
Year               int64
SessionType       object
Location          object
BestQualiTime    float64
Podium             int64
TrackType         object
dtype: object


In [18]:
from sklearn.utils.class_weight import compute_sample_weight

feature_cols = ['BestQualiTime', 'GridPosition', 'TrackType_street', 'TrackType_permanent']  


# One-hot encode TrackType
df_model = pd.get_dummies(df, columns=['TrackType'], dtype=int)

train = df_model[
    (df_model['Year'] < 2025) | 
    ((df_model['Year'] == 2025) & (df_model['Round'] <= 16)) |
    (df_model['Year'] == 2026)  # add this
]
test = df_model[(df_model['Year'] == 2025) & (df_model['Round'] > 16)]
print(train.shape, test.shape)

X_train = train[feature_cols]   
y_train = train['Podium']

X_test = test[feature_cols]
y_test = test['Podium']
w_train = compute_sample_weight(class_weight='balanced', y=y_train)

print(X_train.shape, X_test.shape)

(1303, 12) (160, 12)
(1303, 4) (160, 4)


In [19]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report

model = GradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.96      0.95       136
           1       0.74      0.58      0.65        24

    accuracy                           0.91       160
   macro avg       0.83      0.77      0.80       160
weighted avg       0.90      0.91      0.90       160



In [13]:
import joblib
joblib.dump(model, '../models/model_v4.pkl')
print("Model v4 saved!")

Model v4 saved!
